# Debug C2 — Flowchart & Semantic Blocks

Simula lo scenario C2 (Feed RSS → Dropbox) con il prompt dato,
esegue L1 validation e renderizza Mermaid flowchart + semantic blocks.

In [ ]:
import sys, os, json, re
from pathlib import Path
from IPython.display import HTML, display

# Project root
BASE = Path(os.path.abspath("")).parent.parent
sys.path.insert(0, str(BASE / "src"))

print(f"Project root: {BASE}")

## 1. Carica scenario C2 dal study set enriched

In [ ]:
enriched = json.loads((BASE / "src/evaluation/study_set_enriched.json").read_text(encoding="utf-8"))

SC = None
for sc in enriched.get("non_expert", []):
    if sc["code"] == "C2":
        SC = sc
        break

assert SC is not None, "Scenario C2 non trovato!"

lang = "it"
user_type = "non_expert"
CAT = SC["catalog"][lang]

TRIGGER_INDEX = {t["api_endpoint_slug"]: t for t in CAT["triggers"]}
ACTION_INDEX  = {a["api_endpoint_slug"]: a for a in CAT["actions"]}

print(f"Scenario: {SC['code']}")
print(f"Trigger APIs: {SC['trigger_apis']}")
print(f"Action APIs:  {SC['action_apis']}")
print(f"Services: {SC.get('services', [])}")

## 2. Selezione Ingredient e Setter (tutto tranne Autore)

In [ ]:
# Tutti gli ingredient TRANNE EntryAuthor
sel_ing = set()
for trig_api in SC["trigger_apis"]:
    trig = TRIGGER_INDEX.get(trig_api)
    if trig:
        for ing in trig.get("ingredients", []):
            fck = ing.get("filter_code_key", "")
            if fck and "Author" not in fck:  # escludi autore
                sel_ing.add(fck)

# Tutti i setter
sel_set = set()
for act_api in SC["action_apis"]:
    act = ACTION_INDEX.get(act_api)
    if act:
        ns = act.get("namespace", "")
        for fld in act.get("fields", []):
            method = fld.get("filter_code_method")
            if method:
                sel_set.add(re.sub(r'\(.*\)', '()', method))
            elif fld.get("slug") and ns:
                s = fld["slug"]
                sel_set.add(f"{ns}.set{s[0].upper()}{s[1:]}()")

print("Selected ingredients:")
for g in sorted(sel_ing):
    print(f"  {g}")
print(f"\nSelected setters:")
for s in sorted(sel_set):
    print(f"  {s}")

## 3. Filter Code (simulato — incolla qui il codice generato)

In [ ]:
# Prompt originale:
"""
Quando arriva un nuovo elemento nel feed RSS, salva un file di testo su Dropbox con il titolo e il contenuto dell'articolo. Se l'articolo ha un'immagine, scaricala su Dropbox.
Se la voce non contiene un'immagine, skippa l'azione di download su Dropbox.
Se il contenuto non contiene la parola 'sanremo', skippa  sia l'azione  creazione file di testo su Dropbox e download immagine su Dropbox.
 """

FILTER_CODE = """
if (!Feed.newFeedItem.EntryContent.includes('sanremo')) {
  Dropbox.createTextFileDb.skip();
  Dropbox.addFileFromUrl.skip();
}

Dropbox.createTextFileDb.setFilename(Feed.newFeedItem.EntryTitle + '.txt');
Dropbox.createTextFileDb.setBody(Feed.newFeedItem.EntryContent);
Dropbox.createTextFileDb.setPath('RSS Feed Items');

if (Feed.newFeedItem.EntryImageUrl) {
  Dropbox.addFileFromUrl.setFilename(Feed.newFeedItem.EntryTitle + '.jpg');
  Dropbox.addFileFromUrl.setPath('RSS Feed Images');
  Dropbox.addFileFromUrl.setUrl(Feed.newFeedItem.EntryImageUrl);
} else {
  Dropbox.addFileFromUrl.skip();
}
""".strip()

print(FILTER_CODE)

## 4. L1 Validation

In [ ]:
from code_parsing.feedback import run_l1_validation, L1Report

l1: L1Report = run_l1_validation(
    code=FILTER_CODE,
    trigger_slugs=SC["trigger_apis"],
    action_slugs=SC["action_apis"],
    lang=lang,
)

print(f"Syntax OK: {l1.syntax_ok}")
print(f"Parse error: {l1.parse_error}")
if l1.api_report:
    r = l1.api_report
    print(f"\nAPI Validation:")
    print(f"  Valid getters:   {r.valid_getters}")
    print(f"  Invalid getters: {r.invalid_getters}")
    print(f"  Valid setters:   {r.valid_setters}")
    print(f"  Invalid setters: {r.invalid_setters}")
    print(f"  Skip used:       {r.skip_used}")
    print(f"  Skip available:  {r.skip_available}")

print(f"\nOutcomes: {len(l1.outcomes_raw or [])} paths")
if l1.outcomes_raw:
    from code_parsing.expr import _expr_to_str
    for i, o in enumerate(l1.outcomes_raw):
        cond = _expr_to_str(o['condition']) if o['condition'] else 'True'
        has_skip = bool(o.get('skip_targets'))
        has_setters = bool(o.get('setters'))
        kind = "MIXED" if has_skip and has_setters else ("SKIP" if has_skip else "ACTION")
        print(f"  Path {i} [{kind}]: skip_targets={o.get('skip_targets',[])}, "
              f"setters={len(o['setters'])}, cond={cond[:100]}")

## 5. Display Labels (catalog → human names)

In [ ]:
from code_parsing.catalog_validator import build_display_labels

display_labels = build_display_labels(
    triggers=CAT["triggers"],
    actions=CAT["actions"],
    trigger_slugs=SC["trigger_apis"],
    action_slugs=SC["action_apis"],
    lang=lang,
    services=CAT.get("services", []),
)

print("Getter labels:")
for k, v in sorted(display_labels.get("getter_labels", {}).items()):
    print(f"  {k} -> {v}")
print(f"\nSetter labels:")
for k, v in sorted(display_labels.get("setter_labels", {}).items()):
    print(f"  {k} -> {v}")
print(f"\nSkip labels:")
for k, v in sorted(display_labels.get("skip_labels", {}).items()):
    print(f"  {k} -> {v}")
print(f"\nNamespace names:")
for k, v in sorted(display_labels.get("namespace_names", {}).items()):
    print(f"  {k} -> {v}")
print(f"\nNamespace icons:")
for k, v in sorted(display_labels.get("namespace_icons", {}).items()):
    print(f"  {k} -> {v[:60]}...")

## 6. Mermaid Flowchart

In [ ]:
from code_parsing.flowchart import render_code_flowchart_html

inv_set = None
if l1.api_report and l1.api_report.invalid_setters:
    inv_set = set(l1.api_report.invalid_setters)

mermaid_html, mermaid_height = render_code_flowchart_html(
    FILTER_CODE,
    lang=lang,
    user_type=user_type,
    display_labels=display_labels,
    invalid_setters=inv_set,
)

print(f"Mermaid height: {mermaid_height}px")
if mermaid_html:
    display(HTML(mermaid_html))
else:
    print("Flowchart non disponibile")

## 7. Semantic Blocks (HTML)

In [ ]:
from code_parsing.flowchart import render_flowchart_html

if l1.syntax_ok and l1.outcomes_raw:
    blocks_html = render_flowchart_html(
        l1.outcomes_raw,
        l2_report=None,
        lang=lang,
        user_type=user_type,
        display_labels=display_labels,
    )
    display(HTML(blocks_html))
else:
    print("Semantic blocks non disponibili")

## 8. Debug: Outcomes detail

In [ ]:
from code_parsing.flowchart import _format_condition, _format_setter, _format_skip

if l1.outcomes_raw:
    for i, o in enumerate(l1.outcomes_raw):
        cond = _expr_to_str(o['condition']) if o['condition'] else 'True'
        cond_human = _format_condition(cond, user_type, display_labels, lang)
        has_skip = bool(o.get('skip_targets'))
        has_setters = bool(o.get('setters'))
        kind = "MIXED" if has_skip and has_setters else ("SKIP" if has_skip else "ACTION")
        print(f"=== Path {i} [{kind}] ===")
        print(f"  Raw condition:   {cond[:120]}")
        print(f"  Human condition: {cond_human}")
        if o.get('skip_targets'):
            skip_text = _format_skip(o.get('skip_targets', []), user_type, display_labels, lang)
            print(f"  Skip targets: {o.get('skip_targets', [])}")
            print(f"  Skip display: {skip_text}")
        for j, s in enumerate(o['setters']):
            setter_text = _format_setter(s, user_type, display_labels, lang)
            print(f"  Setter {j}: {s['method']} -> {setter_text}")
        print()